# Working with LLM APIs — Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

These exercises use the Anthropic SDK (`anthropic`). Install with `pip install anthropic`.
API calls require a valid API key in the `ANTHROPIC_API_KEY` environment variable.

**1. Basic client setup and completion.** Initialize the Anthropic client and make a simple completion request.

In [ ]:
import os

def basic_completion(prompt):
    """Initialize Anthropic client and make a simple completion request."""
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("ANTHROPIC_API_KEY not set. Cannot make API call.")
        return None
    
    try:
        from anthropic import Anthropic
        client = Anthropic()
        
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=150,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    except ImportError:
        print("anthropic package not installed. Run: pip install anthropic")
        return None
    except Exception as e:
        print(f"API call failed: {e}")
        return None

result = basic_completion("What is the capital of France? Answer in one sentence.")
if result:
    print(f"Response: {result}")

**2. Token counting exercise.** Use the Anthropic tokenizer to count tokens in various inputs.

In [ ]:
import tiktoken

def count_tokens(text, model="gpt-4"):
    """Count tokens in a text string."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    return len(tokens)

def demonstrate_token_counting():
    """Demonstrate token counting with various inputs."""
    test_inputs = {
        "empty string": "",
        "single word": "hello",
        "sentence": "The quick brown fox jumps over the lazy dog.",
        "paragraph": """Token counting is essential for managing API costs and context
        windows. Each model has its own tokenizer, and different languages and
        code vs prose will tokenize differently. Understanding this helps you
        budget requests and avoid hitting context limits unexpectedly.""",
        "code": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
        "json": '{"name": "John", "age": 30, "city": "New York"}',
    }
    
    print(f"{'Input':<15} {'Chars':>6} {'Tokens':>7} {'Chars/Token':>11}")
    print("-" * 45)
    for label, text in test_inputs.items():
        chars = len(text)
        tokens = count_tokens(text)
        ratio = chars / tokens if tokens > 0 else 0
        print(f"{label:<15} {chars:>6} {tokens:>7} {ratio:>11.1f}")
    
    print("\nToken counting matters because:")
    print("  1. Cost: API providers charge per token")
    print("  2. Context limits: Models have maximum token windows")
    print("  3. Performance: More tokens = slower generation")

demonstrate_token_counting()

**3. Streaming implementation.** Create a function that streams a response and displays it in real-time.

In [ ]:
import os
import sys

def stream_response(prompt):
    """Stream a response from the API and return the complete text."""
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("ANTHROPIC_API_KEY not set.")
        return None
    
    try:
        from anthropic import Anthropic
        client = Anthropic()
        
        full_response = ""
        print("Response: ", end="", flush=True)
        
        with client.messages.stream(
            model="claude-3-5-sonnet-20241022",
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}]
        ) as stream:
            for text in stream.text_stream:
                print(text, end="", flush=True)
                full_response += text
        
        print()  # newline after stream completes
        return full_response
    except ImportError:
        print("anthropic package not installed.")
        return None
    except Exception as e:
        print(f"\nStream error: {e}")
        return None

result = stream_response("Write a haiku about programming.")
if result:
    print(f"\nComplete response ({len(result)} chars): {result}")

**4. Retry mechanism with exponential backoff.** Implement a robust wrapper for API calls that retries on failure.

In [ ]:
import time
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def with_retry(max_attempts=5):
    """Decorator that adds retry logic with exponential backoff."""
    def decorator(func):
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    error_type = type(e).__name__
                    
                    retryable = (
                        "rate" in str(e).lower() or
                        "server" in str(e).lower() or
                        "connection" in str(e).lower() or
                        "timeout" in str(e).lower()
                    )
                    
                    if not retryable or attempt == max_attempts - 1:
                        raise
                    
                    delay = 2 ** attempt  # 1s, 2s, 4s, 8s, 16s
                    logger.warning(
                        f"Attempt {attempt + 1}/{max_attempts} failed: {error_type}. "
                        f"Retrying in {delay}s..."
                    )
                    time.sleep(delay)
            raise last_exception
        return wrapper
    return decorator

# Test with a function that fails twice then succeeds
call_count = 0

@with_retry(max_attempts=5)
def unreliable_api_call():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError("Connection reset by peer")
    return "Success!"

try:
    result = unreliable_api_call()
    print(f"Result: {result}")
except Exception as e:
    print(f"All retries failed: {e}")

**5. Thinking mode demonstration.** Show how to use thinking modes with the Anthropic API.

In [ ]:
import os

def demonstrate_thinking_modes():
    """Demonstrate disabled vs enabled thinking modes."""
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("ANTHROPIC_API_KEY not set. Showing code structure instead.")
        
        print("\n--- Disabled Thinking (default) ---")
        print("""client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=500,
    messages=[{"role": "user", "content": "Solve: 2x + 5 = 15"}]
)""")
        
        print("\n--- Enabled Thinking ---")
        print("""client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=10000,
    thinking={
        "type": "enabled",
        "budget_tokens": 5000
    },
    messages=[{"role": "user", "content": "Solve: 2x + 5 = 15"}]
)""")
        
        print("\nKey differences:")
        print("  - Disabled: Response has only text content")
        print("  - Enabled: Response has thinking blocks + text blocks")
        print("  - Enabled: Requires higher max_tokens (thinking uses tokens)")
        print("  - Enabled: Better for complex reasoning, math, code")
        return
    
    try:
        from anthropic import Anthropic
        client = Anthropic()
        
        prompt = "Solve for x: 2x + 5 = 15"
        
        # Without thinking
        print("\n--- Without Thinking ---")
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}]
        )
        print(f"Response: {response.content[0].text}")
        print(f"Usage: input={response.usage.input_tokens}, output={response.usage.output_tokens}")
        
        # With thinking
        print("\n--- With Thinking ---")
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=10000,
            thinking={
                "type": "enabled",
                "budget_tokens": 5000
            },
            messages=[{"role": "user", "content": prompt}]
        )
        
        for block in response.content:
            if block.type == "thinking":
                print(f"Thinking: {block.thinking}")
            elif block.type == "text":
                print(f"Response: {block.text}")
        print(f"Usage: input={response.usage.input_tokens}, output={response.usage.output_tokens}")
    except Exception as e:
        print(f"Error: {e}")

demonstrate_thinking_modes()

**6. Batch processing with rate limit awareness.** Create a batch processor that handles multiple prompts efficiently.

In [ ]:
import time
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

class RateLimiter:
    """Token bucket rate limiter."""
    def __init__(self, max_per_minute):
        self.max_per_minute = max_per_minute
        self.tokens = max_per_minute
        self.last_refill = time.time()
    
    def acquire(self):
        now = time.time()
        elapsed = now - self.last_refill
        self.tokens = min(self.max_per_minute, self.tokens + elapsed * self.max_per_minute / 60)
        self.last_refill = now
        
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

def batch_process(prompts, max_rpm=50):
    """Process multiple prompts with rate limit awareness."""
    results = [None] * len(prompts)
    rate_limiter = RateLimiter(max_rpm)
    
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("ANTHROPIC_API_KEY not set. Returning mock results.")
        return [f"Response to: {p[:30]}..." for p in prompts]
    
    def process_one(idx, prompt):
        while not rate_limiter.acquire():
            time.sleep(0.1)
        
        try:
            from anthropic import Anthropic
            client = Anthropic()
            response = client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=100,
                messages=[{"role": "user", "content": prompt}]
            )
            return idx, response.content[0].text
        except Exception as e:
            return idx, f"Error: {e}"
    
    start_time = time.time()
    print(f"Processing {len(prompts)} prompts...")
    
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(process_one, i, p): i for i, p in enumerate(prompts)}
        for future in as_completed(futures):
            idx, result = future.result()
            results[idx] = result
            done = sum(1 for r in results if r is not None)
            elapsed = time.time() - start_time
            print(f"  [{done}/{len(prompts)}] {elapsed:.1f}s elapsed")
    
    total_time = time.time() - start_time
    print(f"\nCompleted {len(prompts)} prompts in {total_time:.1f}s")
    return results

# Demo with sample prompts
sample_prompts = [
    "What is 2+2?",
    "Capital of France?",
    "Define 'serendipity'.",
    "Color of the sky?",
    "10 * 10?",
]

results = batch_process(sample_prompts)
for p, r in zip(sample_prompts, results):
    print(f"Q: {p} -> A: {r}")

**7. Caching implementation.** Create a simple caching layer for API responses.

In [ ]:
import hashlib
import json
import time

class APICache:
    """Simple TTL-based cache for LLM API responses."""
    def __init__(self, ttl_seconds=300):
        self.cache = {}
        self.ttl = ttl_seconds
        self.hits = 0
        self.misses = 0
    
    def _make_key(self, prompt, model, **kwargs):
        data = json.dumps({"prompt": prompt, "model": model, **kwargs}, sort_keys=True)
        return hashlib.sha256(data.encode()).hexdigest()
    
    def get(self, prompt, model, **kwargs):
        key = self._make_key(prompt, model, **kwargs)
        if key in self.cache:
            entry = self.cache[key]
            if time.time() - entry["timestamp"] < self.ttl:
                self.hits += 1
                return entry["response"]
            else:
                del self.cache[key]
        self.misses += 1
        return None
    
    def set(self, prompt, model, response, **kwargs):
        key = self._make_key(prompt, model, **kwargs)
        self.cache[key] = {
            "response": response,
            "timestamp": time.time()
        }
    
    def stats(self):
        total = self.hits + self.misses
        hit_rate = self.hits / total * 100 if total > 0 else 0
        return f"Cache: {self.hits} hits, {self.misses} misses ({hit_rate:.1f}% hit rate)"

# Demo
cache = APICache(ttl_seconds=60)

# Simulate cached calls
prompts = ["What is Python?", "What is Python?", "What is Python?", "What is Java?", "What is Python?"]
model = "claude-3-5-sonnet-20241022"

for prompt in prompts:
    cached = cache.get(prompt, model)
    if cached:
        print(f"CACHE HIT: {prompt[:30]}...")
    else:
        response = f"Response to: {prompt}"
        cache.set(prompt, model, response)
        print(f"CACHE MISS: {prompt[:30]}... -> stored")

print(f"\n{cache.stats()}")